# Rental Contract Review Agent

An AI-powered lease advocate that reviews rental contracts through a four-stage pipeline:

1. **Define Standards** — Establish baseline of typical lease terms and legal requirements for the renter's geography
2. **Analyze Clauses** — Assess each clause's favorability and label it (illegal / unfair but legal / unclear / outdated / fair)
3. **Prioritize Findings** — Rank findings by legal risk, financial impact, and negotiability
4. **Generate Report** — Produce a structured Improvement Decision Report with facts, risk, prioritized improvements, and citations

**Architecture:**
- **LangChain** — LLM orchestration, message management, and tool calling
- **LlamaIndex + ChromaDB** — RAG pipeline over gold standard leases, real-world examples, and lease info
- **Web scraping** — State-specific legal research restricted to reputable domains

**Three resource types:**
1. `rag_data/` — Training data (gold standard leases, other leases, info docs) indexed into ChromaDB
2. **User lease ingestion** — Portal integration point for uploaded lease files or pasted text
3. **Web search** — Domain-allowlisted scraping for jurisdiction-specific tenant law

> **CLI execution note:** To run this notebook from the command line without overwriting the source file, use:
> ```
> jupyter execute --output=rental_contracts/agent_executed.ipynb rental_contracts/agent.ipynb
> ```

In [ ]:
# ============================================================
# ENVIRONMENT BOOTSTRAP
# ============================================================
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab")
    from google.colab import drive
    drive.mount("/content/drive")
    get_ipython().system('pip install -e /content/drive/MyDrive/BUSN30135/busn30135_labs')

try:
    from course_utils import env_setup
except ImportError:
    print("Package not found. Attempting to reload...")
    import site
    site.main()
    if '' not in sys.path:
        sys.path.insert(0, '')
    try:
        from course_utils import env_setup
    except ImportError:
        print("CRITICAL: Package still not found. Please RESTART THE RUNTIME/KERNEL.")
        raise

DIRECTORY_LABS = env_setup.setup_lab("agent")
DIRECTORY_LAB = os.path.join(DIRECTORY_LABS, "agent")

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import hashlib
import json
import time
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

# --- LangChain (lab_02 pattern) ---
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool

# --- LlamaIndex RAG (lab_05 pattern) ---
from llama_index.core import (
    SimpleDirectoryReader,
    StorageContext,
    VectorStoreIndex,
    Settings,
    Document,
)
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI as LlamaIndexOpenAI
from llama_index.vector_stores.chroma import ChromaVectorStore

# --- ChromaDB ---
import chromadb

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# --- Model Configuration ---
OPENAI_MODEL_ID = "gpt-4o-mini"

# Primary LLM for the agent pipeline (LangChain)
llm = ChatOpenAI(model=OPENAI_MODEL_ID, temperature=0)

# LlamaIndex LLM and embedding model for RAG
Settings.llm = LlamaIndexOpenAI(model=OPENAI_MODEL_ID)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# --- Path Configuration ---
DIR_RAG_DATA = os.path.join(DIRECTORY_LABS, "rag_data")
DIR_CHROMADB = os.path.join(DIRECTORY_LAB, "chromadb")

# --- User Lease Input ---
# >>> PORTAL INTEGRATION POINT <<<
# Set one of these before running the pipeline:
#   USER_LEASE_PATH = "path/to/uploaded/lease.pdf"
#   USER_LEASE_TEXT = "raw text of the lease..."
# The web portal should set one of these variables, or call
# ingest_user_lease() directly with the file path or text.
USER_LEASE_PATH = None  # e.g., "/uploads/tenant_lease.pdf"
USER_LEASE_TEXT = None   # e.g., raw text string from OCR or paste

# --- Geography Configuration ---
# >>> USER: Set the renter's location for jurisdiction-aware analysis <<<
RENTER_STATE = "PLACEHOLDER_STATE"  # e.g., "Illinois"
RENTER_CITY = "PLACEHOLDER_CITY"    # e.g., "Chicago"

# --- Allowed Domains for Web Scraping ---
ALLOWED_DOMAINS = [
    "nolo.com",
    "law.cornell.edu",
    "findlaw.com",
    "justia.com",
    "hud.gov",
    "usa.gov",
    "tenant.net",
    "landlordology.com",
    "avail.co",
]

# ============================================================
# PROMPTS
# ============================================================

# --- Stage 1: fixed, not tested in prompt lab ---
PROMPT_DEFINE_STANDARDS = """
You are a tenant rights legal expert. Your job is to establish the legal and contractual
standards that apply to the lease being reviewed.

Using the tools available to you:
1. Retrieve examples of fair, gold-standard lease language for comparison.
2. Search for tenant rights laws and landlord obligations applicable to the renter's jurisdiction.

Then produce a structured standards framework covering:
- Key statutes and local ordinances that apply (e.g. security deposit limits, notice requirements)
- Provisions that are legally required in leases in this jurisdiction
- Provisions that are prohibited or unenforceable by law
- What fair, balanced language looks like for common clause types (rent, deposits, entry, repairs, termination)
- Red flag language patterns that commonly disadvantage tenants

ACCURACY RULES — follow these strictly:
- Only assert something is "illegal" or "required by law" if you can cite a specific statute or ordinance.
- Distinguish carefully between: (a) rules that apply IF a landlord does something (e.g. IF a security
  deposit is collected, THEN it must be held in a specific account) vs. (b) things a landlord is
  required to do regardless. Do NOT conflate these.
- If you are unsure about a specific legal requirement, say so rather than guessing.

Be specific to the renter's city and state. This framework will be used to evaluate every clause
in the lease, so accuracy is more important than comprehensiveness.
""".strip()

# --- Stage 2: winner from 03_prompt_lab.ipynb (v3_expert) ---
PROMPT_ANALYZE_CLAUSES = """
You are a tenant rights attorney with 20 years of experience reviewing residential leases.
A renter has hired you to protect their interests. Your professional reputation depends on catching every problem.

Audit every single clause in this lease. For each clause:
1. Consider what the standards framework says about this type of clause.
2. Ask: does this language protect the tenant, expose them to risk, or is it neutral?
3. Assign a label and severity, then explain your reasoning.

Use exactly this format for each clause:

**Clause**: [clause name or short description]
**Label**: [illegal / unfair but legal / unclear or ambiguous / outdated / fair]
**Severity**: [integer 1-10, where 10 is most harmful to the tenant]
**Explanation**: [1-2 sentences citing specific laws or standard benchmarks]

Label definitions:
- illegal: violates applicable housing law; unenforceable — ONLY use this if you can cite a specific law
- unfair but legal: legal but significantly disadvantages the tenant
- unclear or ambiguous: vague or exploitable language
- outdated: no longer reflects current law or practice
- fair: balanced and reasonable

ACCURACY RULES — follow these strictly:
- Do NOT label a clause "illegal" unless you can name the specific statute or ordinance it violates.
- Pay close attention to the direction of legal obligations. Example: Chicago/Illinois law regulates
  HOW security deposits must be handled IF collected — it does not require landlords to collect one.
  A lease that waives a security deposit is not illegal; a lease that improperly withholds one may be.
- When in doubt about legality, use "unclear or ambiguous" rather than "illegal".
- Severity must still reflect the real risk to the tenant regardless of label. A clause that is
  probably illegal but can't be confirmed should still get severity 7-9 if it exposes the tenant
  to serious harm.

Be exhaustive. Missing a problematic clause is a professional failure.
""".strip()

# --- Stage 3: fixed, not tested in prompt lab ---
PROMPT_PRIORITIZE_FINDINGS = """
You are a tenant advocate. Given the clause analysis results below, rank all non-fair findings
by how urgently the tenant should act on them.

Tier assignment — use SEVERITY as the primary driver:
- CRITICAL: severity 8-10 (major financial risk or likely legal violation)
- HIGH: severity 5-7 (significantly disadvantages tenant, worth negotiating hard)
- MEDIUM: severity 3-4 (worth raising but not deal-breakers)
- LOW: severity 1-2 (minor issues, note but deprioritize)

For each finding output exactly:
  Tier | Clause Name | Why it's ranked here | Negotiable? (yes/no/maybe)

List CRITICAL items first, then HIGH, MEDIUM, LOW. Skip clauses labeled "fair".
""".strip()

# --- Stage 4: winner from 03_prompt_lab.ipynb (v3_expert) ---
PROMPT_GENERATE_REPORT = """
You are a tenant rights advocate writing a report for a renter who is not a legal expert.
Your job is to be their champion — clear, empowering, and actionable.

Generate a complete Improvement Decision Report with these sections:

1. **Executive Summary**: 2-3 sentences. Give the tenant a clear bottom line —
   is this lease safe to sign, risky, or problematic?
2. **Key Facts**: Bullet list of material terms (rent, deposit, lease term, notice periods, etc.)
3. **Risk Assessment**: All non-fair clauses grouped by tier (CRITICAL → HIGH → MEDIUM → LOW).
   For each: clause name, label, severity score, one-sentence risk description.
4. **Prioritized Improvements**: For each CRITICAL and HIGH item:
   - The problem in plain language
   - Why it matters financially or legally
   - Exact negotiation language the tenant can use
5. **Educational Notes**: Define legal terms used (joint and several liability, holdover tenant,
   habitability, abatement, etc.) in plain language.
6. **Advocacy Resources**: Note any local tenant rights organizations relevant to the tenant's location.
7. **Citations**: All laws, ordinances, and standards referenced.

Write at an 8th-grade reading level. Tone: empowering, not overwhelming.

ACCURACY RULES:
- Only cite a law as violated if the clause analysis explicitly identified it as illegal with a citation.
- Do not introduce new legal claims that weren't in the clause analysis.
""".strip()

## RAG Pipeline: Knowledge Base Construction

Build three ChromaDB-backed vector indices from `rag_data/`:
- **Gold Standard Leases** — Fair, standard lease templates from housing authorities and bar associations
- **Other Leases** — Real-world lease examples for comparison (various cities and formats)
- **Lease Info** — Educational resources on lease red flags and illegal clauses

Separate collections enable targeted retrieval: the agent queries gold standards for fairness benchmarks, other leases for real-world comparisons, and lease info for general guidance.

In [ ]:
# ============================================================
# RAG PIPELINE — BUILD VECTOR INDICES (lab_05 pattern)
# ============================================================

def build_vector_index(directory_path, collection_name):
    """Build a ChromaDB-backed vector index from a directory of documents."""
    documents = SimpleDirectoryReader(directory_path, recursive=True).load_data()
    print(f"Loaded {len(documents)} document(s) from {directory_path}")

    chroma_client = chromadb.PersistentClient(path=DIR_CHROMADB)
    chroma_collection = chroma_client.get_or_create_collection(collection_name)

    vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)

    index = VectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context,
        embed_model=Settings.embed_model,
    )
    print(f"Vector index created for collection: {collection_name}")
    return index


# Build indices from rag_data/ subfolders
index_gold_standard = build_vector_index(
    os.path.join(DIR_RAG_DATA, "gold_standard_leases"), "gold_standard_leases"
)
index_other_leases = build_vector_index(
    os.path.join(DIR_RAG_DATA, "other_leases"), "other_leases"
)
index_lease_info = build_vector_index(
    os.path.join(DIR_RAG_DATA, "info"), "lease_info"
)

# Create query engines
query_engine_gold_standard = index_gold_standard.as_query_engine()
query_engine_other_leases = index_other_leases.as_query_engine()
query_engine_lease_info = index_lease_info.as_query_engine()

In [ ]:
# ============================================================
# USER LEASE INGESTION
# >>> PORTAL INTEGRATION POINT <<<
# The web portal should call ingest_user_lease() with either:
#   - file_path: path to an uploaded PDF/DOCX/TXT file
#   - raw_text: pasted or OCR'd lease text
# For standalone notebook use, set USER_LEASE_PATH or
# USER_LEASE_TEXT in the Configuration cell above.
# ============================================================

def ingest_user_lease(file_path=None, raw_text=None):
    """Ingest a user-supplied lease from file or raw text.

    Args:
        file_path: Path to a lease file (PDF, DOCX, TXT).
        raw_text: Raw lease text as a string.

    Returns:
        str: The extracted lease text.
    """
    # Use arguments if provided, otherwise fall back to config variables
    file_path = file_path or USER_LEASE_PATH
    raw_text = raw_text or USER_LEASE_TEXT

    if file_path:
        file_path = Path(file_path)
        documents = SimpleDirectoryReader(
            input_files=[str(file_path)]
        ).load_data()
        text = "\n\n".join([doc.text for doc in documents])
        print(f"Ingested lease from file: {file_path.name} ({len(text):,} chars)")
        return text
    elif raw_text:
        print(f"Ingested lease from raw text ({len(raw_text):,} chars)")
        return raw_text
    else:
        raise ValueError(
            "No lease provided. Set USER_LEASE_PATH or USER_LEASE_TEXT "
            "in the config cell, or pass file_path/raw_text to this function."
        )

## Agent Tools

LangChain `@tool` functions that the LLM can call during any pipeline stage to retrieve information from the RAG knowledge base.

In [ ]:
# ============================================================
# AGENT TOOLS (lab_02 @tool + bind_tools pattern)
# ============================================================

import re

@tool
def retrieve_gold_standard_clauses(query: str) -> str:
    """Retrieve examples of fair, standard lease language from gold standard
    lease templates. Use this to compare a clause against what is considered
    typical and fair in rental contracts.

    Args:
        query: A description of the lease clause or topic to find standard language for.
    """
    response = query_engine_gold_standard.query(query)
    return str(response)


@tool
def retrieve_other_lease_examples(query: str) -> str:
    """Retrieve examples from real-world lease agreements for comparison.
    Use this to see how other leases handle a specific clause or provision.

    Args:
        query: A description of the lease clause or topic to compare across leases.
    """
    response = query_engine_other_leases.query(query)
    return str(response)


@tool
def retrieve_lease_info(query: str) -> str:
    """Retrieve educational information about lease red flags, illegal clauses,
    and best practices for tenants. Use this when you need general guidance
    on what to look for in a lease.

    Args:
        query: A question about lease red flags, illegal clauses, or tenant rights.
    """
    response = query_engine_lease_info.query(query)
    return str(response)


# --- URL patterns for direct fetching from allowed domains ---
# Each entry maps a domain to URL templates that cover tenant-rights pages.
# The {topic} and {state} placeholders are filled at query time.
DOMAIN_URL_PATTERNS = {
    "nolo.com": [
        "https://www.nolo.com/legal-encyclopedia/tenants-rights-{state}.html",
        "https://www.nolo.com/legal-encyclopedia/{topic}.html",
    ],
    "findlaw.com": [
        "https://www.findlaw.com/state/{state}/landlord-tenant-law.html",
        "https://www.findlaw.com/realestate/landlord-tenant-law/{topic}.html",
    ],
    "justia.com": [
        "https://www.justia.com/real-estate/landlord-tenant/{topic}/",
    ],
    "law.cornell.edu": [
        "https://www.law.cornell.edu/wex/landlord-tenant_law",
    ],
    "hud.gov": [
        "https://www.hud.gov/topics/rental_assistance",
    ],
    "tenant.net": [
        "https://www.tenant.net/rights/{topic}/",
    ],
    "avail.co": [
        "https://www.avail.co/education/laws/{state}-landlord-tenant-law",
    ],
    "landlordology.com": [
        "https://www.landlordology.com/{state}/landlord-tenant-law/",
    ],
    "usa.gov": [
        "https://www.usa.gov/landlord-tenant-disputes",
    ],
}


def _fetch_page_text(url, max_chars=3000):
    """Fetch a URL and extract readable text with BeautifulSoup."""
    try:
        resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=8)
        if resp.status_code != 200:
            return None
        soup = BeautifulSoup(resp.text, "html.parser")
        # Remove script/style noise
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()
        text = soup.get_text(separator="\n", strip=True)
        # Collapse blank lines
        text = re.sub(r'\n{3,}', '\n\n', text)
        return text[:max_chars] if text.strip() else None
    except requests.RequestException:
        return None


@tool
def search_legal_web(query: str, state: str = "") -> str:
    """Search reputable legal websites for state-specific tenant rights and
    housing laws. Fetches pages directly from pre-approved domains (no
    Google scraping, no API keys needed).

    Args:
        query: The legal topic to search for (e.g., 'security deposit limits').
        state: The US state to search laws for (e.g., 'Illinois'). Defaults to RENTER_STATE.
    """
    if not state:
        state = RENTER_STATE
    state_slug = state.lower().replace(" ", "-")
    topic_slug = query.lower().replace(" ", "-")
    results = []

    for domain in ALLOWED_DOMAINS:
        patterns = DOMAIN_URL_PATTERNS.get(domain, [])
        for pattern in patterns:
            url = pattern.format(state=state_slug, topic=topic_slug)
            text = _fetch_page_text(url)
            if text:
                results.append(f"[{domain}] ({url})\n{text}")
                break  # one hit per domain is enough

    if results:
        return "\n\n---\n\n".join(results)
    return (
        f"No results found for '{query}' in {state}. "
        f"Searched {len(ALLOWED_DOMAINS)} allowed domains: "
        f"{', '.join(ALLOWED_DOMAINS)}"
    )


# Bind tools to the LLM
TOOLS = [
    retrieve_gold_standard_clauses,
    retrieve_other_lease_examples,
    retrieve_lease_info,
    search_legal_web,
]
llm_with_tools = llm.bind_tools(TOOLS)

In [ ]:
# ============================================================
# CACHING UTILITY (lab_02 pattern)
# ============================================================

CACHE_FILE = Path("agent_cache.json")


def load_cache():
    """Load cached LLM responses from file."""
    if CACHE_FILE.exists():
        return json.loads(CACHE_FILE.read_text())
    return {}


def save_cache(cache):
    """Save LLM responses cache to file."""
    CACHE_FILE.write_text(json.dumps(cache, indent=2))


def get_cache_key(stage_name, content_hash):
    """Generate a cache key from stage name and content."""
    key_str = f"{stage_name}:{content_hash}"
    return hashlib.sha256(key_str.encode()).hexdigest()[:16]

In [ ]:
# ============================================================
# OUTPUT PARSING (lab_04 pattern)
# ============================================================

import re

def parse_clause_analysis(output_text):
    """Parse structured clause analysis from LLM output.

    Handles flexible LLM formatting — numbered lists, bold markers,
    headers, or plain text. Splits the output into clause sections
    and extracts fields by keyword matching.

    Returns:
        list[dict]: Each dict has keys: clause_name, label, severity,
                    explanation, raw_output
    """
    # --- Split into clause sections ---
    # Match boundaries like "1. ", "## ", "### ", "**Clause", "- **Clause", "Clause:"
    boundary = re.compile(
        r'(?=(?:^|\n)'               # start of line
        r'(?:'
        r'\d+[\.\)]\s'               # numbered list: "1. " or "1) "
        r'|#{2,}\s'                   # markdown header: "## " or "### "
        r'|-\s*\*\*'                  # bullet bold: "- **"
        r'|\*\*Clause'               # bold Clause label
        r'|Clause\s*[\d#]*\s*:'      # "Clause:" or "Clause 1:"
        r'))',
        re.IGNORECASE | re.MULTILINE,
    )

    positions = [m.start() for m in boundary.finditer(output_text)]
    if not positions:
        # No recognisable structure — return single raw entry
        return [_extract_fields(output_text)]

    sections = []
    for i, start in enumerate(positions):
        end = positions[i + 1] if i + 1 < len(positions) else len(output_text)
        section = output_text[start:end].strip()
        if section:
            sections.append(section)

    # --- Extract fields from each section ---
    findings = []
    for section in sections:
        parsed = _extract_fields(section)
        # Skip sections that look like preamble (no clause name found)
        if parsed["clause_name"] or parsed["label"]:
            findings.append(parsed)

    return findings


def _extract_fields(section_text):
    """Extract clause_name, label, severity, explanation from a text section."""
    lines = [ln.strip() for ln in section_text.splitlines() if ln.strip()]

    result = {
        "clause_name": None,
        "label": None,
        "severity": None,
        "explanation": None,
        "raw_output": section_text,
    }

    # Ordered list — most specific first so "unfair but legal" is matched
    # before "fair" (which is a substring of "unfair but legal").
    valid_labels = [
        "illegal",
        "unfair but legal",
        "unclear or ambiguous",
        "unclear",
        "ambiguous",
        "outdated",
        "fair",
    ]

    for line in lines:
        low = line.lower()
        # Strip markdown bold/italic wrappers for matching
        clean = re.sub(r'[\*#\-]+', '', line).strip()

        # --- Clause name ---
        if result["clause_name"] is None:
            m = re.search(r'clause[^:]*:\s*(.+)', low)
            if m:
                result["clause_name"] = re.sub(r'[\*#]+', '', m.group(1)).strip().title()
                continue
            # First line of a numbered entry is often the clause name
            if re.match(r'^\d+[\.\)]', line) and result["clause_name"] is None:
                name = re.sub(r'^\d+[\.\)]\s*', '', clean)
                if name and "label" not in name.lower() and "severity" not in name.lower():
                    result["clause_name"] = name.strip(":").title()
                    continue

        # --- Label ---
        if result["label"] is None and "label" in low and ":" in line:
            val = line.split(":", 1)[1].strip().strip("*").lower()
            for lbl in valid_labels:
                if lbl in val:
                    result["label"] = lbl
                    break
            if result["label"] is None and val:
                result["label"] = val  # keep whatever the LLM said

        # --- Severity ---
        if result["severity"] is None and "severity" in low and ":" in line:
            nums = re.findall(r'[\d]+(?:\.[\d]+)?', line.split(":", 1)[1])
            if nums:
                try:
                    result["severity"] = float(nums[0])
                except ValueError:
                    pass

        # --- Explanation ---
        if result["explanation"] is None and "explanation" in low and ":" in line:
            result["explanation"] = line.split(":", 1)[1].strip().strip("*")

    return result

In [ ]:
# ============================================================
# TOOL-CALLING LOOP HELPER (with caching)
# ============================================================

def call_llm_with_tools(messages, max_tool_rounds=3, stage_name=None):
    """Call the LLM and handle any tool calls it makes.

    Implements the tool-calling loop from lab_02:
    1. Check cache — if hit, return cached result immediately
    2. Send messages to LLM with bound tools
    3. If LLM makes tool calls, execute them
    4. Feed tool results back and repeat
    5. Cache the final result before returning

    Args:
        messages: List of LangChain message objects.
        max_tool_rounds: Maximum number of tool-calling rounds.
        stage_name: Pipeline stage name for cache key (e.g., "define_standards").

    Returns:
        The final AIMessage content (str).
    """
    # --- Cache check ---
    cache = load_cache()
    if stage_name:
        content_str = "|".join(m.content for m in messages if hasattr(m, "content"))
        content_hash = hashlib.sha256(content_str.encode()).hexdigest()[:32]
        cache_key = get_cache_key(stage_name, content_hash)
        if cache_key in cache:
            print(f"  Cache HIT for {stage_name}")
            return cache[cache_key]
    else:
        cache_key = None

    # --- LLM call with tool loop ---
    tool_map = {t.name: t for t in TOOLS}

    for round_num in range(max_tool_rounds):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            result = response.content
            break

        # Execute each tool call
        for tool_call in response.tool_calls:
            tool_fn = tool_map[tool_call["name"]]
            tool_result = tool_fn.invoke(tool_call["args"])
            messages.append(
                ToolMessage(content=str(tool_result), tool_call_id=tool_call["id"])
            )
            print(f"  Tool called: {tool_call['name']}")
    else:
        # Exhausted rounds — final call without tools
        response = llm.invoke(messages)
        result = response.content

    # --- Cache store ---
    if cache_key:
        cache[cache_key] = result
        save_cache(cache)
        print(f"  Cache STORED for {stage_name}")

    return result

## Analysis Pipeline

Four-stage pipeline that processes a rental contract from raw text to a structured Improvement Decision Report.

In [ ]:
# ============================================================
# STAGE 1: DEFINE STANDARDS
# ============================================================

def define_standards(contract_text):
    """Establish baseline standards using Prompt 1 + RAG.

    Uses gold standard leases and legal references to define what is
    typical, fair, and legally required for the renter's geography.
    """
    messages = [
        SystemMessage(content=PROMPT_DEFINE_STANDARDS),
        HumanMessage(content=(
            f"Renter's Location: {RENTER_CITY}, {RENTER_STATE}\n\n"
            f"Lease Contract to Review:\n{contract_text[:8000]}\n\n"
            "Using the tools available to you, retrieve relevant legal standards "
            "and gold standard lease language for this jurisdiction. Then establish "
            "the baseline standards against which this lease will be evaluated."
        )),
    ]

    result = call_llm_with_tools(messages, stage_name="define_standards")
    print("Stage 1 (Define Standards) complete.")
    return result

In [ ]:
# ============================================================
# STAGE 2: ANALYZE CLAUSES
# ============================================================

def analyze_clauses(contract_text, standards):
    """Assess each clause's favorability and label it.

    Labels: illegal / unfair but legal / unclear or ambiguous / outdated / fair
    Severity: 1 (minor) to 10 (critical)

    Returns:
        tuple: (list of parsed findings dicts, raw analysis text)
    """
    messages = [
        SystemMessage(content=PROMPT_ANALYZE_CLAUSES),
        HumanMessage(content=(
            f"Renter's Location: {RENTER_CITY}, {RENTER_STATE}\n\n"
            f"Standards Framework:\n{standards}\n\n"
            f"Lease Contract:\n{contract_text}\n\n"
            "Analyze each clause in this lease. For each clause, provide:\n"
            "- **Clause**: name/description\n"
            "- **Label**: illegal / unfair but legal / unclear or ambiguous / outdated / fair\n"
            "- **Severity**: score from 1 (minor) to 10 (critical)\n"
            "- **Explanation**: why this clause received this label\n\n"
            "Use the tools to verify against legal standards and gold standard language."
        )),
    ]

    raw_analysis = call_llm_with_tools(messages, stage_name="analyze_clauses")

    # Parse the structured output (format-flexible parser)
    findings = parse_clause_analysis(raw_analysis)

    print(f"Stage 2 (Analyze Clauses) complete. Found {len(findings)} clauses.")
    return findings, raw_analysis

In [ ]:
# ============================================================
# STAGE 3: PRIORITIZE FINDINGS
# ============================================================

def prioritize_findings(raw_analysis, standards):
    """Rank findings by importance considering legal risk, financial
    impact, and negotiability."""
    messages = [
        SystemMessage(content=PROMPT_PRIORITIZE_FINDINGS),
        HumanMessage(content=(
            f"Renter's Location: {RENTER_CITY}, {RENTER_STATE}\n\n"
            f"Standards Framework:\n{standards}\n\n"
            f"Clause Analysis Results:\n{raw_analysis}\n\n"
            "Rank the findings above by priority. Consider:\n"
            "- Legal risk (illegal clauses first)\n"
            "- Financial impact on the renter\n"
            "- Likelihood of successful negotiation\n"
            "- Long-term consequences\n"
            "Provide a numbered priority list with justification for the ranking."
        )),
    ]

    result = call_llm_with_tools(messages, stage_name="prioritize_findings")
    print("Stage 3 (Prioritize Findings) complete.")
    return result

In [ ]:
# ============================================================
# STAGE 4: GENERATE REPORT
# ============================================================

def generate_report(contract_text, standards, raw_analysis, prioritized_findings):
    """Produce a structured Improvement Decision Report.

    Includes: executive summary, facts, risk assessment, prioritized
    improvements, negotiation tips, educational notes, advocacy
    resources, and citations.
    """
    messages = [
        SystemMessage(content=PROMPT_GENERATE_REPORT),
        HumanMessage(content=(
            f"Renter's Location: {RENTER_CITY}, {RENTER_STATE}\n\n"
            f"Standards Framework:\n{standards}\n\n"
            f"Clause Analysis:\n{raw_analysis}\n\n"
            f"Prioritized Findings:\n{prioritized_findings}\n\n"
            "Generate a complete Improvement Decision Report. Include:\n"
            "1. Executive Summary of the lease review\n"
            "2. Facts: Key terms and conditions found\n"
            "3. Risk Assessment: Legal and financial risks identified\n"
            "4. Prioritized Improvements: Ranked list of clauses to negotiate\n"
            "5. Negotiation Tips: Actionable advice for each priority item\n"
            "6. Educational Notes: Explain key legal concepts "
            "(joint and several liability, abatement, termination, holdover, etc.)\n"
            "7. Advocacy Resources: Use the search tool to find local resources\n"
            "8. Citations: Specific laws, ordinances, and standards referenced\n"
        )),
    ]

    result = call_llm_with_tools(messages, stage_name="generate_report")
    print("Stage 4 (Generate Report) complete.")
    return result

## Main Execution: Run the Full Pipeline

In [ ]:
# ============================================================
# LOAD THE CONTRACT
# ============================================================
# For testing, use one of the leases from rag_data/other_leases/.
# In production, the portal sets USER_LEASE_PATH or USER_LEASE_TEXT
# in the config cell, or calls ingest_user_lease() directly.
if USER_LEASE_PATH is None and USER_LEASE_TEXT is None:
    # Default to a sample lease for testing
    USER_LEASE_PATH = os.path.join(DIR_RAG_DATA, "other_leases", "Chicago Sublease.pdf")

contract_text = ingest_user_lease()

# ============================================================
# RUN THE FOUR-STAGE PIPELINE
# ============================================================
print("\n" + "=" * 60)
print("STAGE 1: Define Standards")
print("=" * 60)
standards = define_standards(contract_text)

print("\n" + "=" * 60)
print("STAGE 2: Analyze Clauses")
print("=" * 60)
findings, raw_analysis = analyze_clauses(contract_text, standards)

print("\n" + "=" * 60)
print("STAGE 3: Prioritize Findings")
print("=" * 60)
prioritized = prioritize_findings(raw_analysis, standards)

print("\n" + "=" * 60)
print("STAGE 4: Generate Report")
print("=" * 60)
report = generate_report(contract_text, standards, raw_analysis, prioritized)

## Improvement Decision Report

In [ ]:
from IPython.display import Markdown, display

display(Markdown(report))

In [ ]:
# ============================================================
# EXPORT RESULTS
# ============================================================

# Save the full report as markdown
with open("improvement_decision_report.md", "w") as f:
    f.write(report)
print("Report saved to improvement_decision_report.md")

# Save structured findings as CSV
if findings:
    df_findings = pd.DataFrame(findings)
    df_findings.to_csv("clause_findings.csv", index=False)
    print("Structured findings saved to clause_findings.csv")
    display(df_findings.drop(columns=["raw_output"]))

In [ ]:
# ============================================================
# EXPORT INTERMEDIATE OUTPUTS FOR PROMPT LAB
# Run this after the full pipeline to enable fast prompt testing
# in evaluation/03_prompt_lab.ipynb without re-running the pipeline.
# ============================================================

import json as _json
from pathlib import Path as _Path

_prompt_test_inputs = {
    "contract_text": contract_text,
    "standards": standards,
    "raw_analysis": raw_analysis,
    "prioritized": prioritized,
    "renter_city": RENTER_CITY,
    "renter_state": RENTER_STATE,
}

_output_path = _Path("../evaluation/prompt_test_inputs.json")
_output_path.parent.mkdir(exist_ok=True)
_output_path.write_text(_json.dumps(_prompt_test_inputs, indent=2), encoding="utf-8")
print(f"Prompt lab inputs saved to: {_output_path.resolve()}")